# GDACS global hazard dashboard

A 90-day snapshot of **every GDACS hazard type** worldwide: a faceted world map (one panel per hazard) coloured by alert level, plus a table of the current **Red** alerts.

GDACS is public — no credentials, no extra to install. `download()` returns a pyramids `FeatureCollection` (a `geopandas.GeoDataFrame`).

## Setup

First the imports. `matplotlib` and `pandas` are used by the plotting and summary cells; `earthlens` provides the unified `EarthLens` entry point. We also create the output directory the backend writes into.

In [ ]:
import datetime as dt
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from digitalearth.web import WebMap

from earthlens.core import EarthLens

# Off the system drive; override with EARTHLENS_CACHE.
OUT_DIR = Path(os.environ.get('EARTHLENS_CACHE', 'D:/earthlens-cache')) / 'gdacs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

### The request window

We look back 90 days from today and list the six GDACS hazard codes we want, with friendly names for the map titles.

In [ ]:
TODAY = dt.date.today()
START = (TODAY - dt.timedelta(days=90)).strftime('%Y-%m-%d')
END = TODAY.strftime('%Y-%m-%d')

HAZARDS = ['EQ', 'TC', 'FL', 'VO', 'WF', 'DR']
HAZARD_NAMES = {
    'EQ': 'Earthquake',
    'TC': 'Tropical cyclone',
    'FL': 'Flood',
    'VO': 'Volcano',
    'WF': 'Wildfire',
    'DR': 'Drought',
}
print('window:', START, 'to', END)

## Pull every hazard for the window

A single request returns all six hazard types. We build the request first, then call `download()` on its own line so each step is easy to read and re-run. The call is wrapped so the notebook stays safe under nbval-lax when run offline.

In [ ]:
alerts = None
try:
    dashboard = EarthLens(
        variables=HAZARDS,
        data_source='gdacs',
        start=START,
        end=END,
        aoi=[-180, -90, 180, 90],
        alert_level=['Green', 'Orange', 'Red'],
        path=OUT_DIR,
    )
    alerts = dashboard.download()
    print(f'{len(alerts)} alerts')
    print(alerts['hazard_type'].value_counts().to_dict())
except Exception as exc:
    print(f'live GDACS query skipped (offline?): {type(exc).__name__}: {exc}')

## Faceted world map — one panel per hazard

Each panel shows that hazard's alerts coloured by level (green / orange / red).

In [ ]:
if alerts is not None and len(alerts):
    colours = {'Green': 'green', 'Orange': 'orange', 'Red': 'red'}
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
    for ax, code in zip(axes.ravel(), HAZARDS):
        subset = alerts[alerts['hazard_type'] == code]
        ax.set_title(f'{HAZARD_NAMES[code]} ({len(subset)})')
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        if len(subset):
            subset.plot(
                ax=ax,
                color=subset['alert_level'].map(colours).fillna('grey'),
                markersize=18,
                alpha=0.6,
            )
    fig.suptitle(f'GDACS alerts by hazard, {START} to {END}')
    fig.tight_layout()
    plt.show()

## One shareable map — Digital-Earth's web tier

The six panels above are static and coastline-less: an alert is a dot at a longitude and a latitude, with
nothing behind it to say which country it landed in. For a hazard dashboard that is the one thing a reader
wants.

`digitalearth.web.WebMap` renders the same `GeoDataFrame` as MapLibre GL layers over a real basemap and
`save()` writes a **self-contained HTML file** — pan, zoom and click any alert to read its name, country and
score, with no server and no Python at the other end. That is the tier's point: something you can send to
someone who does not have the notebook.

In [ ]:
if alerts is not None and len(alerts):
    hazard_map = (
        WebMap(height=520)
        .basemap('CartoDark')
        .points(alerts, column='alert_score', cmap='inferno', size=6, opacity=0.85)
        .tooltip(['name', 'country', 'hazard_type', 'alert_level'])
        .navigation()
        .scale_bar()
    )
    # Also written as a standalone page, for sending to someone without the notebook.
    print('standalone copy:', hazard_map.save(str(OUT_DIR / 'gdacs_hazards.html')))

In [ ]:
# A real MapLibre map: drag to pan, scroll to zoom, click an alert for its name,
# country, hazard type and level. The widget state is saved with the notebook, so
# it stays live without a kernel.
hazard_map if alerts is not None and len(alerts) else 'no alerts to map'

## The current Red alerts

Red is GDACS's highest impact level. These are the alerts most likely to need a coordinated response.

In [ ]:
if alerts is not None and len(alerts):
    red = alerts[alerts['alert_level'] == 'Red'].sort_values(
        'from_date', ascending=False
    )
    print(f'{len(red)} Red alerts in the window')
    cols = ['from_date', 'hazard_type', 'name', 'country', 'alert_score', 'glide']
    display(red[cols].head(20))

## Alert-level breakdown per hazard

A compact cross-tab of how many alerts of each level each hazard produced.

In [ ]:
if alerts is not None and len(alerts):
    crosstab = pd.crosstab(alerts['hazard_type'], alerts['alert_level'])
    for level in ['Green', 'Orange', 'Red']:
        if level not in crosstab.columns:
            crosstab[level] = 0
    display(crosstab[['Green', 'Orange', 'Red']])